In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 48.025,
	"longitude": -98.1171,
	"start_date": "2021-04-03",
	"end_date": "2026-04-03",
	"daily": ["weather_code", "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "rain_sum", "precipitation_sum", "precipitation_hours", "wind_gusts_10m_max", "wind_speed_10m_max", "sunshine_duration"],
	"timezone": "America/Chicago",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(3).ValuesAsNumpy()
daily_rain_sum = daily.Variables(4).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(5).ValuesAsNumpy()
daily_precipitation_hours = daily.Variables(6).ValuesAsNumpy()
daily_wind_gusts_10m_max = daily.Variables(7).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(8).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(9).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["rain_sum"] = daily_rain_sum
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["precipitation_hours"] = daily_precipitation_hours
daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
# print("\nDaily data\n", daily_dataframe)

rain_df = daily_dataframe[daily_dataframe["rain_sum"] != 0]
rain_df.head()


Coordinates: 48.0492057800293°N -98.08651733398438°E
Elevation: 462.0 m asl
Timezone: b'America/Chicago'b'GMT-5'
Timezone difference to GMT+0: -18000s


,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,rain_sum,precipitation_sum,precipitation_hours,wind_gusts_10m_max,wind_speed_10m_max,sunshine_duration
5,2021-04-08 00:00:00+00:00,51.0,6.385417,8.85,4.20,1.3,1.3,7.0,38.880001,20.268991,16090.037109
6,2021-04-09 00:00:00+00:00,51.0,6.347918,10.35,2.55,1.3,1.3,7.0,56.519997,30.752090,27910.947266
9,2021-04-12 00:00:00+00:00,73.0,0.093750,5.00,-4.65,0.2,3.0,16.0,54.719997,29.522196,138.183746
13,2021-04-16 00:00:00+00:00,51.0,3.404167,7.70,0.35,0.1,0.1,1.0,32.760002,17.581125,15546.582031
15,2021-04-18 00:00:00+00:00,73.0,1.391666,4.75,-3.20,0.1,1.4,3.0,59.399998,32.042019,8602.098633
